# Neural Identifier Training with Particle Filters - Lorenz System Identification

In [2]:
import numpy as np
import plotly.graph_objects as go

In [3]:
# ============================================================
# 1) True nonlinear system (Lorenz Chaotic System)
# ============================================================
def plant_dynamics(x, u, sigma=5.0, rho=14.0, beta=4.0/3.0):
    """
    Continuous dynamics for Lorenz chaotic system: x = [x1, x2, x3].
    Returns x_dot.
    
    The Lorenz system equations:
    dx1/dt = sigma * (x2 - x1) + u1
    dx2/dt = x1 * (rho - x3) - x2 + u2  
    dx3/dt = x1 * x2 - beta * x3 + u3
    
    where:
    x1, x2, x3: state variables
    u1, u2, u3: control inputs (can be zero for autonomous system)
    sigma: Prandtl number (typically 10)
    rho: Rayleigh number (typically 28) - scaled to 14 for stability
    beta: geometric parameter (typically 8/3) - scaled to 4/3 for stability
    """
    x1, x2, x3 = x
    
    # Ensure u is properly formatted as 3-element array
    if isinstance(u, (list, np.ndarray)):
        if len(u) >= 3:
            u1, u2, u3 = u[0], u[1], u[2]
        elif len(u) == 1:
            u1, u2, u3 = u[0], 0.0, 0.0
        else:
            u1, u2, u3 = 0.0, 0.0, 0.0
    else:
        u1, u2, u3 = float(u), 0.0, 0.0
    
    # Lorenz chaotic system dynamics (scaled parameters for numerical stability)
    x1_dot = sigma * (x2 - x1) + u1
    x2_dot = x1 * (rho - x3) - x2 + u2
    x3_dot = x1 * x2 - beta * x3 + u3
    
    return np.array([x1_dot, x2_dot, x3_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='mixed', process_noise_std=0.05, 
          parameter_variation=0.02, sensor_bias=[0.0, 0.0, 0.0]):
    """
    One Euler step of the discrete plant with realistic Lorenz system disturbances.
    
    Args:
        x_k: current state [x1, x2, x3]
        u_k: control input [u1, u2, u3] or scalar u1
        dt: time step
        process_noise_type: type of noise ('mixed', 'gaussian', 'laplacian')
        process_noise_std: standard deviation of process noise
        parameter_variation: Lorenz parameter variations
        sensor_bias: systematic biases in measurements [x1_bias, x2_bias, x3_bias]
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Realistic Lorenz system disturbances
    
    # 1. Parameter variations (affects system dynamics)
    param_noise = parameter_variation * np.array([
        np.random.randn() * np.abs(x_kp1[1] - x_kp1[0]),  # sigma variation effect
        np.random.randn() * np.abs(x_kp1[0]),              # rho variation effect  
        np.random.randn() * np.abs(x_kp1[0] * x_kp1[1])   # beta variation effect
    ])
    
    # 2. Control-dependent noise (increases with control magnitude)
    if isinstance(u_k, (list, np.ndarray)):
        control_magnitude = np.linalg.norm(u_k)
    else:
        control_magnitude = np.abs(u_k)
    control_noise_factor = 1 + 0.1 * control_magnitude
    
    # 3. Mixed process noise (combination of different noise types)
    if process_noise_type == 'mixed':
        # Gaussian component (main noise)
        gaussian_noise = np.random.normal(0, process_noise_std * control_noise_factor, size=x_kp1.shape)
        # Impulse noise (occasional large disturbances)
        impulse_prob = 0.02  # 2% chance of impulse noise
        impulse_noise = np.zeros_like(x_kp1)
        if np.random.rand() < impulse_prob:
            impulse_noise = np.random.normal(0, process_noise_std * 3, size=x_kp1.shape)
        # Laplacian component (heavy-tailed noise)
        laplacian_noise = np.random.laplace(0, process_noise_std * 0.3, size=x_kp1.shape)
        
        total_noise = gaussian_noise + impulse_noise + laplacian_noise
    elif process_noise_type == 'laplacian':
        total_noise = np.random.laplace(0, process_noise_std * control_noise_factor, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std * control_noise_factor
        total_noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        total_noise = np.random.normal(0, process_noise_std * control_noise_factor, size=x_kp1.shape)
    
    # 4. Systematic biases (drift, calibration errors)
    bias_noise = np.array(sensor_bias) * dt
    
    # 5. Measurement quantization effects
    measurement_resolution = 0.001  # 0.001 unit resolution
    quantization_noise = measurement_resolution * (np.random.rand(3) - 0.5)
    
    # Combine all disturbances
    x_kp1 += param_noise + total_noise + bias_noise + quantization_noise
    
    # 6. State bounds for numerical stability (prevent extreme values)
    x_kp1 = np.clip(x_kp1, -50, 50)
    
    return x_kp1

def generate_realistic_trajectory(t, trajectory_type='sine'):
    """
    Generate realistic control inputs for Lorenz system.
    
    Args:
        t: time value
        trajectory_type: 'sine', 'square', 'step', 'mixed', 'chaotic_stabilization'
    
    Returns:
        u: [u1, u2, u3] control inputs for x1, x2, x3 states
    """
    if trajectory_type == 'sine':
        # Sinusoidal inputs with different frequencies
        u1 = 0.5 * np.sin(0.3 * t)
        u2 = 0.3 * np.sin(0.5 * t + np.pi/4)
        u3 = 0.2 * np.sin(0.7 * t + np.pi/2)
        return np.array([u1, u2, u3])
    
    elif trajectory_type == 'square':
        # Square wave inputs
        period = 10.0  # 10 second period
        u1 = 0.4 if (t % period) < (period / 2) else -0.4
        u2 = 0.3 if ((t + period/4) % period) < (period / 2) else -0.3
        u3 = 0.2 if ((t + period/2) % period) < (period / 2) else -0.2
        return np.array([u1, u2, u3])
    
    elif trajectory_type == 'step':
        # Step inputs
        if t < 8.0:
            u = np.array([0.2, 0.0, 0.0])
        elif t < 16.0:
            u = np.array([0.0, 0.2, 0.0])
        elif t < 24.0:
            u = np.array([0.0, 0.0, 0.2])
        else:
            u = np.array([0.1, 0.1, 0.1])
        return u
    
    elif trajectory_type == 'chaotic_stabilization':
        # Control designed to stabilize chaotic behavior
        # Simple proportional control (requires state feedback - simplified here)
        target = np.array([0.0, 0.0, 20.0])  # Target point in Lorenz attractor
        k = 0.1  # Control gain
        u1 = -k * np.sin(0.2 * t)
        u2 = -k * np.cos(0.15 * t) 
        u3 = -k * np.sin(0.1 * t)
        return np.array([u1, u2, u3])
    
    else:  # 'mixed' - combination of different behaviors
        # Mixed trajectory with different phases
        phase = (t % 32.0) / 32.0  # 32-second cycles
        
        if phase < 0.25:  # Sine wave
            u1 = 0.3 * np.sin(0.4 * t)
            u2 = 0.2 * np.sin(0.6 * t)
            u3 = 0.1 * np.sin(0.8 * t)
        elif phase < 0.5:  # Step input
            u1, u2, u3 = 0.2, 0.0, 0.0
        elif phase < 0.75:  # Negative sine
            u1 = -0.2 * np.sin(0.3 * t)
            u2 = -0.15 * np.sin(0.5 * t)
            u3 = -0.1 * np.sin(0.7 * t)
        else:  # Damped oscillation
            decay = np.exp(-0.1 * (t % 8))
            u1 = 0.2 * np.sin(0.8 * t) * decay
            u2 = 0.15 * np.sin(1.2 * t) * decay
            u3 = 0.1 * np.sin(1.6 * t) * decay
        
        return np.array([u1, u2, u3])

In [4]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    # z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input=None):
    """
    Features for a 3-state Lorenz chaotic system with control input:
    x = [x1, x2, x3], u = [u1, u2, u3] or scalar u1
    
    z = [x1, x2, x3, x1*x2, x1*x3, x2*x3, x1^2, x2^2, x3^2, u, 1]
    
    This feature vector captures the essential nonlinear interactions 
    in the Lorenz system while maintaining reasonable dimensionality.
    """
    x1, x2, x3 = x_est[0], x_est[1], x_est[2]
    
    # Apply aggressive clipping to prevent numerical overflow
    x1 = np.clip(x1, -20, 20)
    x2 = np.clip(x2, -20, 20) 
    x3 = np.clip(x3, -20, 20)
    
    # Basic features - direct state terms
    features = [
        x1, x2, x3,                          # Direct state terms
        x1 * x2,                             # Cross term x1*x2 (important for Lorenz)
        x1 * x3,                             # Cross term x1*x3
        x2 * x3,                             # Cross term x2*x3
        x1**2, x2**2, x3**2,                 # Quadratic terms
    ]
    
    # Add control input features if available
    if u_input is not None:
        if isinstance(u_input, (list, np.ndarray)) and len(u_input) >= 1:
            u_control = u_input[0]            # Use first control input
        else:
            u_control = float(u_input)        # Scalar input
        
        # Clip control input to prevent overflow
        u_control = np.clip(u_control, -10, 10)
        features.append(u_control)            # Direct control term
    else:
        features.append(0.0)                  # Zero placeholder if no control input
    
    # Add bias term
    features.append(1.0)                      # Bias term
    
    return np.array(features)


def RHONN_predict(x_state_for_z, w_neuron, u_input=None):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z, u_input)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

In [5]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, filter's own estimate at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: use filter's own estimate at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_input)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance with regularization
            P_pred = self.P[i] + self.Q[i]
            
            # Add small regularization to maintain positive definiteness
            P_pred += np.eye(self.num_weights_per_neuron) * 1e-8

            # Innovation covariance (scalar) with improved numerical stability
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i
            
            # Clip innovation to prevent extreme updates
            e_i = np.clip(e_i, -10.0, 10.0)

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update with adaptive learning rate
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))
            self.weights[i] += adaptive_eta * K_i * e_i

            # Joseph form covariance update for better numerical stability
            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-6

In [6]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=None, R_std=None, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        
        # Handle Q_std (process noise) - can be scalar or list/array per neuron
        if Q_std is None:
            self.Q_std = [0.05] * num_neurons
        elif isinstance(Q_std, (int, float)):
            self.Q_std = [Q_std] * num_neurons
        else:
            self.Q_std = list(Q_std) if len(Q_std) == num_neurons else [0.05] * num_neurons
            
        # Handle R_std (measurement noise) - can be scalar or list/array per neuron  
        if R_std is None:
            self.R_std = [0.1] * num_neurons
        elif isinstance(R_std, (int, float)):
            self.R_std = [R_std] * num_neurons
        else:
            self.R_std = list(R_std) if len(R_std) == num_neurons else [0.1] * num_neurons
            
        # Compute R_var for each neuron
        self.R_var = [r**2 for r in self.R_std]
        
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : filter's own estimate at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        u_input: control input at time k (for mobile robot)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)
        z = construct_z_vector(x_state_for_z, u_input)  # (num_features,)

        # 1) Predict: random walk on weights with state-specific Q_std
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std[i]

        # 2) Update: importance weights with Gaussian likelihood using state-specific R_var
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability using state-specific R_var
            ll = -0.5 * (innov**2) / self.R_var[i]
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

    def get_parameters_info(self):
        """Return information about the PF parameters for each state."""
        state_names = ['x', 'y', 'theta']
        info = {}
        for i in range(self.num_neurons):
            name = state_names[i] if i < len(state_names) else f'state_{i}'
            info[name] = {
                'Q_std': self.Q_std[i],
                'R_std': self.R_std[i],
                'R_var': self.R_var[i]
            }
        return info

In [7]:
# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        
        # UKF parameters
        self.alpha = alpha  # Spread of sigma points (typically 1e-4 to 1)
        self.beta = beta    # Prior knowledge about distribution (2 for Gaussian)
        self.kappa = kappa if kappa is not None else 3 - num_weights_per_neuron
        
        # Calculate lambda and weights for sigma points
        self.n = num_weights_per_neuron
        self.lambda_ = alpha**2 * (self.n + self.kappa) - self.n
        
        # Weights for mean and covariance computation
        self.Wm = np.zeros(2 * self.n + 1)
        self.Wc = np.zeros(2 * self.n + 1)
        
        self.Wm[0] = self.lambda_ / (self.n + self.lambda_)
        self.Wc[0] = self.lambda_ / (self.n + self.lambda_) + (1 - alpha**2 + beta)
        
        for i in range(1, 2 * self.n + 1):
            self.Wm[i] = 1.0 / (2 * (self.n + self.lambda_))
            self.Wc[i] = 1.0 / (2 * (self.n + self.lambda_))

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance):
        """Generate sigma points for UKF."""
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        
        # First sigma point is the mean
        sigma_points[0] = mean
        
        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + self.lambda_) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + self.lambda_)
        
        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        
        return sigma_points

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One UKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, filter's own estimate at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: use filter's own estimate at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_input)  # shape (num_features,)

        for i in range(self.num_neurons):
            # --- Prediction Step ---
            # Generate sigma points for current weights
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i])
            
            # Predict sigma points (weights don't change in prediction step for RHONN)
            predicted_sigma_points = sigma_points.copy()
            
            # Predict mean and covariance
            predicted_mean = np.sum(self.Wm[:, np.newaxis] * predicted_sigma_points, axis=0)
            
            predicted_cov = self.Q[i].copy()
            for j in range(2 * self.n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[j] * np.outer(diff, diff)
            
            # --- Measurement Update ---
            # Transform sigma points through measurement function
            measurement_sigma_points = np.zeros(2 * self.n + 1)
            for j in range(2 * self.n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)
            
            # Predicted measurement mean
            predicted_measurement = np.sum(self.Wm * measurement_sigma_points)
            
            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * self.n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += self.Wc[j] * diff_meas**2
            
            # Cross-covariance
            cross_cov = np.zeros(self.n)
            for j in range(2 * self.n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += self.Wc[j] * diff_state * diff_meas
            
            # Kalman gain
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12
            K = cross_cov / innovation_cov
            
            # Innovation
            innovation = chi_kp1[i] - predicted_measurement
            
            # State update
            self.weights[i] = predicted_mean + self.eta * K * innovation
            
            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.n) * 1e-6

In [8]:
# ============================================================
# 5) Simulation Main Loop
# ============================================================

# --- Simulation settings ---
n_steps = 1500
dt = 0.02
t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

process_noise_type = 'mixed'  # 'mixed' | 'laplacian' | 'uniform' | 'gaussian'
process_noise_std = 0.01
parameter_variation = 0.01
sensor_bias = [0.001, 0.0005, 0.0003]  # Small systematic biases [x1, x2, x3]

# --- True system init ---
x_true = np.zeros((n_steps, 3))
x_true[0] = [1.0, 1.0, 1.0]  # Initial conditions for Lorenz system [x1, x2, x3]

# --- Control trajectory ---
trajectory_type = 'sine'  # 'sine', 'square', 'step', 'mixed', 'chaotic_stabilization'

# --- RHONN config ---
num_neurons = 3  # Three states for Lorenz system [x1, x2, x3]
num_features = 11  # Updated feature vector size for 3D Lorenz system + control
num_weights_per_neuron = num_features

# --- Common initial weights for fair comparison ---
# np.random.seed(12345)  # (optional) reproducibility of initial weights
common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
print("Common Initial Weights:")
for i, w in enumerate(common_initial_weights):
    print(f"  Neuron {i}: {w}")

# --- EKF --- (Tuned parameters for Lorenz system)
ekf_trainer = EKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=1e-4, R_init=5e-3, P_init=1.0, eta=0.5
)
x_hat_ekf = np.zeros((n_steps, 3))
x_hat_ekf[0] = x_true[0]

# --- UKF ---
ukf_trainer = UKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=1e-4, R_init=5e-3, P_init=1.0, eta=0.9,
    alpha=1e-2, beta=2.0  # UKF-specific parameters for Lorenz system
)
x_hat_ukf = np.zeros((n_steps, 3))
x_hat_ukf[0] = x_true[0]

# --- PF ---
n_particles = 300  # Particles for 3-state Lorenz system

# State-specific noise parameters: [x1, x2, x3]
Q_std_per_state = [0.04, 0.06, 0.08]  # Process noise for Lorenz states
R_std_per_state = [0.03, 0.04, 0.06]  # Measurement noise for Lorenz states

pf_trainer = PF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    n_particles=n_particles,
    initial_weights=common_initial_weights,
    Q_std=Q_std_per_state, R_std=R_std_per_state, 
    ess_threshold=n_particles / 2  # ESS < N/2
)

# Display PF parameters for verification
pf_params = pf_trainer.get_parameters_info()
print("\nParticle Filter Parameters per State:")
for state, params in pf_params.items():
    print(f"  {state}: Q_std={params['Q_std']:.3f}, R_std={params['R_std']:.3f}, R_var={params['R_var']:.6f}")

# Force identical particle initialization if desired:
def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
    for i in range(pf_trainer_instance.num_neurons):
        pf_trainer_instance.particles[i] = np.tile(
            common_weights_list[i], (pf_trainer_instance.n_particles, 1)
        )
        pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

x_hat_pf = np.zeros((n_steps, 3))
x_hat_pf[0] = x_true[0]

print("\nStarting Lorenz system simulation...")
for k in range(n_steps - 1):
    # ---- 1) Generate control input and evolve true system -> k+1 ----
    t_current = k * dt
    u_current = generate_realistic_trajectory(t_current, trajectory_type)
    x_true[k+1] = plant(x_true[k], u_current, dt, process_noise_type, process_noise_std, parameter_variation, sensor_bias)

    # ---- 2) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
    ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ekf[k], x_hat_previous=x_hat_ekf[k], u_input=u_current)

    x_state_for_z_ekf = np.copy(x_hat_ekf[k])
    # Series-parallel configuration for Lorenz system
    x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0], u_current)  # x1
    x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1], u_current)  # x2
    x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[2], u_current)  # x3

    # ---- 3) UKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
    ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ukf[k], x_hat_previous=x_hat_ukf[k], u_input=u_current)

    x_state_for_z_ukf = np.copy(x_hat_ukf[k])
    # Series-parallel configuration for Lorenz system
    x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0], u_current)  # x1
    x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1], u_current)  # x2
    x_hat_ukf[k+1, 2] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[2], u_current)  # x3

    # ---- 4) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
    pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_pf[k], x_hat_previous=x_hat_pf[k], u_input=u_current)

    pf_weight_estimates = pf_trainer.get_estimate()
    x_state_for_z_pf = np.copy(x_hat_pf[k])
    # Series-parallel configuration for Lorenz system
    x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0], u_current)   # x1
    x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1], u_current)   # x2
    x_hat_pf[k+1, 2] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[2], u_current)   # x3

    if k % (n_steps // 10) == 0:
        print(f"Simulation progress: {k/n_steps*100:.1f}%")

print("Simulation finished.")

Common Initial Weights:
  Neuron 0: [-0.43622121  0.28016528  0.10463315 -0.41749814 -0.16773769 -0.22995018
 -0.158459    0.15406318  0.26991956 -0.2145648   0.33626028]
  Neuron 1: [ 0.43976081 -0.02939071 -0.28579091 -0.04615508  0.32550411 -0.47393845
 -0.25193662 -0.16799853 -0.09369799 -0.01449044  0.205383  ]
  Neuron 2: [ 0.27551032  0.2644207   0.12906046  0.41198589  0.24122019  0.17406259
  0.4648964  -0.47183421 -0.17946171  0.18966251 -0.06896715]

Particle Filter Parameters per State:
  x: Q_std=0.040, R_std=0.030, R_var=0.000900
  y: Q_std=0.060, R_std=0.040, R_var=0.001600
  theta: Q_std=0.080, R_std=0.060, R_var=0.003600

Starting Lorenz system simulation...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 40.0%
Simulation progress: 50.0%
Simulation progress: 40.0%
Simulation progress: 50.0%
Simulation progress: 60.0%
Simulation progress:

In [9]:
# ============================================================
    # 6) Results & plots for Lorenz System Identification
# ============================================================
mse_x1_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)     # Lorenz state x1
mse_x2_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)     # Lorenz state x2
mse_x3_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)     # Lorenz state x3

mse_x1_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)     # Lorenz state x1
mse_x2_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)     # Lorenz state x2
mse_x3_ukf = np.mean((x_true[:, 2] - x_hat_ukf[:, 2])**2)     # Lorenz state x3

mse_x1_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)       # Lorenz state x1
mse_x2_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)       # Lorenz state x2
mse_x3_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)       # Lorenz state x3

print(f"\nFinal EKF-RHONN Weights:")
for i in range(3):
    state_names = ['x1', 'x2', 'x3']
    print(f"  Neuron {i+1} ({state_names[i]}): {ekf_trainer.weights[i]}")

print(f"\nFinal UKF-RHONN Weights:")
for i in range(3):
    state_names = ['x1', 'x2', 'x3']
    print(f"  Neuron {i+1} ({state_names[i]}): {ukf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(3):
    state_names = ['x1', 'x2', 'x3']
    print(f"  Neuron {i+1} ({state_names[i]}): {pf_estimates[i]}")

print("\n--- Performance Comparison (MSE) for Lorenz System ---")
print(f"EKF MSE x1:      {mse_x1_ekf:.6f}")
print(f"EKF MSE x2:      {mse_x2_ekf:.6f}")
print(f"EKF MSE x3:      {mse_x3_ekf:.6f}")
print(f"UKF MSE x1:      {mse_x1_ukf:.6f}")
print(f"UKF MSE x2:      {mse_x2_ukf:.6f}")
print(f"UKF MSE x3:      {mse_x3_ukf:.6f}")
print(f"PF  MSE x1:      {mse_x1_pf:.6f}")
print(f"PF  MSE x2:      {mse_x2_pf:.6f}")
print(f"PF  MSE x3:      {mse_x3_pf:.6f}")

states_info = [
    {'idx': 0, 'var': 'x1', 'desc': 'Lorenz State X1', 'y_label': 'x₁',
     'chi': 'χ₁ (True x₁)', 'x': 'x₁ (Est.)'},
    {'idx': 1, 'var': 'x2', 'desc': 'Lorenz State X2', 'y_label': 'x₂',
     'chi': 'χ₂ (True x₂)', 'x': 'x₂ (Est.)'},
    {'idx': 2, 'var': 'x3', 'desc': 'Lorenz State X3', 'y_label': 'x₃',
     'chi': 'χ₃ (True x₃)', 'x': 'x₃ (Est.)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                            name=state_info['chi'], line=dict(color='black', width=2))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                        name=f"{state_info['x']} (EKF)", line=dict(dash='dash', color='blue'))
    trace_ukf = go.Scatter(x=t_history, y=x_hat_ukf[:, i], mode='lines',
                        name=f"{state_info['x']} (UKF)", line=dict(dash='dashdot', color='green'))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                        name=f"{state_info['x']} (PF)", line=dict(dash='dot', color='red'))

    fig = go.Figure([trace_plant, trace_ekf, trace_ukf, trace_pf])
    fig.update_layout(
        title=f'Pendulum RHONN Identification for {state_info["var"]} ({state_info["desc"]})',
        xaxis_title='Time (s)',
        yaxis_title=state_info['y_label'],
        legend=dict(x=0, y=1, orientation='h'),
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig.show()

# Errors for both states
error_theta_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_thetadot_ekf = x_true[:, 1] - x_hat_ekf[:, 1]

error_theta_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
error_thetadot_ukf = x_true[:, 1] - x_hat_ukf[:, 1]

error_theta_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_thetadot_pf = x_true[:, 1] - x_hat_pf[:, 1]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ekf, mode='lines',
                        name=f'EKF Error θ (MSE={mse_theta_ekf:.6f})', opacity=0.7, line=dict(color='blue')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ukf, mode='lines',
                        name=f'UKF Error θ (MSE={mse_theta_ukf:.6f})', opacity=0.7, line=dict(color='green')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_pf, mode='lines',
                        name=f'PF Error θ (MSE={mse_theta_pf:.6f})', opacity=0.7, line=dict(color='red')))
fig2.add_trace(go.Scatter(x=t_history, y=error_thetadot_ekf, mode='lines',
                        name=f'EKF Error θ̇ (MSE={mse_thetadot_ekf:.6f})', opacity=0.7, line=dict(color='blue', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_thetadot_ukf, mode='lines',
                        name=f'UKF Error θ̇ (MSE={mse_thetadot_ukf:.6f})', opacity=0.7, line=dict(color='green', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_thetadot_pf, mode='lines',
                        name=f'PF Error θ̇ (MSE={mse_thetadot_pf:.6f})', opacity=0.7, line=dict(color='red', dash='dot')))
fig2.update_layout(
    title='Lorenz System RHONN Identification Errors (EKF vs UKF vs PF)',
    xaxis_title='Time (s)',
    yaxis_title='Error',
    legend=dict(x=0, y=1, orientation='h'),
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()

# 3D Lorenz Attractor Visualization
fig_lorenz = go.Figure()
fig_lorenz.add_trace(go.Scatter3d(x=x_true[:, 0], y=x_true[:, 1], z=x_true[:, 2], 
                                 mode='lines',
                                 name='True Lorenz Attractor',
                                 line=dict(color='black', width=4)))
fig_lorenz.add_trace(go.Scatter3d(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1], z=x_hat_ekf[:, 2],
                                 mode='lines',
                                 name='EKF Estimation',
                                 line=dict(color='blue', width=3)))
fig_lorenz.add_trace(go.Scatter3d(x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1], z=x_hat_ukf[:, 2],
                                 mode='lines',
                                 name='UKF Estimation',
                                 line=dict(color='green', width=3)))
fig_lorenz.add_trace(go.Scatter3d(x=x_hat_pf[:, 0], y=x_hat_pf[:, 1], z=x_hat_pf[:, 2],
                                 mode='lines',
                                 name='PF Estimation',
                                 line=dict(color='red', width=3)))
# Add start and end markers
fig_lorenz.add_trace(go.Scatter3d(x=[x_true[0, 0]], y=[x_true[0, 1]], z=[x_true[0, 2]], 
                                 mode='markers',
                                 name='Start', 
                                 marker=dict(color='green', size=8, symbol='diamond')))
fig_lorenz.add_trace(go.Scatter3d(x=[x_true[-1, 0]], y=[x_true[-1, 1]], z=[x_true[-1, 2]], 
                                 mode='markers',
                                 name='End', 
                                 marker=dict(color='red', size=8, symbol='square')))
fig_lorenz.update_layout(
    title='Lorenz System Attractor - 3D State Space Comparison (EKF vs UKF vs PF)',
    scene=dict(
        xaxis_title='x₁',
        yaxis_title='x₂',
        zaxis_title='x₃',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
    ),
    font=dict(size=12),
    showlegend=True
)
fig_lorenz.show()

# Determine which filter has the lowest total MSE (sum of theta, theta_dot)
mse_total_ekf = mse_theta_ekf + mse_thetadot_ekf
mse_total_ukf = mse_theta_ukf + mse_thetadot_ukf
mse_total_pf = mse_theta_pf + mse_thetadot_pf

mse_totals = {'EKF': mse_total_ekf, 'UKF': mse_total_ukf, 'PF': mse_total_pf}
best_filter = min(mse_totals, key=mse_totals.get)

print(f"\nBest overall performance: {best_filter} (lowest total MSE: {mse_totals[best_filter]:.6f})")


Final EKF-RHONN Weights:
  Neuron 1 (x1): [-0.58141131  0.60275881  0.48995844  0.20737377  0.0749381  -0.03045119
 -0.24139663  0.01573267  0.02007878  0.51582444  0.68846658]
  Neuron 2 (x2): [ 0.11564383  0.22198168  0.61062421  1.719103   -0.52699498 -1.24363282
 -1.80605593  0.49775477  0.46659616  1.1744588   0.74345081]
  Neuron 3 (x3): [-0.03230437  0.10269482  0.32838165  0.00367942  0.0093277  -0.01082403
 -0.00692686  0.01946371  0.02171978  0.38496202  0.48003804]

Final UKF-RHONN Weights:
  Neuron 1 (x1): [-0.64025105  0.82384917 -0.64292676 -1.06248462 -0.0705447  -0.05754491
  0.60584822  0.49426202 -0.0460542  -0.42234513  0.48185654]
  Neuron 2 (x2): [-0.05451977  0.51203569 -0.47842988 -0.11681314 -0.10041761  0.0381178
  0.0706556   0.16338225 -0.01209198 -0.13085554  0.34785429]
  Neuron 3 (x3): [ 0.87055178 -0.04663937  2.50188526  0.79148057  0.20508519  0.0310526
 -0.45578755 -0.57468899  0.03173576  0.43676748 -0.07970948]

Final PF-RHONN Weight Estimates:
  Ne

NameError: name 'mse_theta_ekf' is not defined